# Koemi-3HIP HERM — T4 overnight experiment

This notebook is an executable measurement harness for the 128-expert / 6-active deterministic MoE configuration. It uses the English `HuggingFaceTB/smol-smoltalk` conversation corpus: practical instruction following and dialogue, intentionally without a math-heavy objective. The six experts are selected by a causal content hash; this is a reproducible dispatch baseline, not a learned semantic router.

The run is wall-clock bounded (default five hours), resumable, and writes checkpoints plus JSON/CSV/PNG evidence under `/content/koemi-3hip-results`. No T4 result is claimed until this notebook is executed on Colab.

In [ ]:
# Colab setup
!git clone -q https://github.com/Koemi-AI/Koemi-3HIP.git /content/Koemi-3HIP || (cd /content/Koemi-3HIP && git pull -q)
%cd /content/Koemi-3HIP
%pip install -q datasets matplotlib pandas
import json, math, os, random, time, csv, sys
sys.path.insert(0, '/content/Koemi-3HIP/src')
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from koemi.configuration.settings import ModelSettings
from koemi.data.adapters import ShareGptRecordAdapter
from koemi.model.execution import ExecutionMode
from koemi.model.network import KoemiModel
from koemi.training.dataset import CausalByteDataset, create_training_loader, IGNORE_TARGET_ID
from koemi.training.objective import calculate_training_objective, token_cross_entropy
SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'This experiment is intended for a Colab T4 GPU.'
print({'device': torch.cuda.get_device_name(0), 'torch': torch.__version__, 'cuda': torch.version.cuda})
RESULTS = Path('/content/koemi-3hip-results'); RESULTS.mkdir(exist_ok=True)


In [ ]:
# Dataset: English conversational reasoning and instruction following
# The card reports 460k English training rows, Apache-2.0, and no advanced-math focus.
raw = load_dataset('HuggingFaceTB/smol-smoltalk', split='train', streaming=True)
adapter = ShareGptRecordAdapter()
records = []
for index, row in enumerate(raw):
    if index >= 60000: break
    messages = row.get('messages', row.get('conversations'))
    if not isinstance(messages, list): continue
    try:
        records.append(adapter.adapt({'id': str(index), 'conversations': messages}, str(index)))
    except Exception:
        continue
if len(records) < 1000: raise RuntimeError(f'Only {len(records)} valid conversations loaded')
random.shuffle(records)
split = max(1, int(len(records) * 0.05))
valid_records, train_records = tuple(records[:split]), tuple(records[split:])
SEQ_LEN = 512
train_dataset = CausalByteDataset(train_records, SEQ_LEN)
valid_dataset = CausalByteDataset(valid_records, SEQ_LEN)
print({'records': len(records), 'train_chunks': len(train_dataset), 'valid_chunks': len(valid_dataset), 'sequence_length': SEQ_LEN})


In [ ]:
# T4-oriented model and loader settings
MODEL_SETTINGS = ModelSettings(embedding_size=128, memory_features=16, local_memory_size=16, salience_memory_size=16, expert_count=128, expert_top_k=6, scan_chunk=32, ablation='no_refine')
model = KoemiModel(MODEL_SETTINGS).to(DEVICE)
parameter_count = sum(p.numel() for p in model.parameters())
# T4 has 16 GB. Start conservatively; reduce only if the OOM probe says so.
BATCH_SIZE = 4
train_loader = create_training_loader(train_dataset, BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, prefetch_factor=2)
valid_loader = create_training_loader(valid_dataset, BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True, prefetch_factor=2)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01, betas=(0.9, 0.95))
scaler = torch.cuda.amp.GradScaler(enabled=True)
AMP_DTYPE = torch.float16
print({'parameters': parameter_count, 'experts': 128, 'active_experts_per_token': 6, 'batch_size': BATCH_SIZE, 'amp': str(AMP_DTYPE)})
# Deterministic top-k dispatch is checked before spending the night.
probe = torch.randint(0, 256, (1, 8), device=DEVICE)
with torch.inference_mode():
    probe_output = model(probe, execution_mode=ExecutionMode.PARALLEL)
assert probe_output.active_expert_indices.shape[-1] == 6
assert all(x == 6 for x in (probe_output.active_expert_indices >= 0).sum(dim=-1).flatten().tolist())
del probe, probe_output; torch.cuda.empty_cache()


In [ ]:
# Five-hour resumable training loop with per-step instrumentation
CHECKPOINT = RESULTS / 'koemi-3hip-t4-moe-checkpoint.pt'
STEP_LOG = RESULTS / 'steps.jsonl'
BUDGET_SECONDS = 5 * 60 * 60
SAVE_EVERY = 250
start_step = 0
if CHECKPOINT.exists():
    payload = torch.load(CHECKPOINT, map_location=DEVICE)
    model.load_state_dict(payload['model']); optimizer.load_state_dict(payload['optimizer'])
    scaler.load_state_dict(payload['scaler']); start_step = int(payload['step'])
    print('resumed', start_step)
history = []
started = time.perf_counter(); step = start_step; tokens_seen = 0; model.train(); optimizer.zero_grad(set_to_none=True)
while time.perf_counter() - started < BUDGET_SECONDS:
    for batch in train_loader:
        if time.perf_counter() - started >= BUDGET_SECONDS: break
        input_ids = batch['input_ids'].to(DEVICE, non_blocking=True)
        target_ids = batch['target_ids'].to(DEVICE, non_blocking=True)
        thinking_mask = batch['thinking_mask'].to(DEVICE, non_blocking=True)
        supervised = int((target_ids != IGNORE_TARGET_ID).sum())
        if not supervised: continue
        t0 = time.perf_counter()
        with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
            output = model(input_ids, execution_mode=ExecutionMode.PARALLEL)
            objective = calculate_training_objective(output, target_ids, thinking_mask, 1.0)
        scaler.scale(objective.total_loss).backward()
        scaler.unscale_(optimizer); grad_norm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0))
        scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
        step += 1; tokens_seen += supervised
        elapsed = time.perf_counter() - t0
        row = {'step': step, 'loss_nats': float(objective.total_loss.detach()), 'bpb': float(objective.total_loss.detach()) / math.log(2), 'tokens': tokens_seen, 'tokens_per_sec': supervised / max(elapsed, 1e-9), 'step_seconds': elapsed, 'grad_norm': grad_norm, 'gpu_allocated_mb': torch.cuda.memory_allocated() / 2**20, 'gpu_reserved_mb': torch.cuda.memory_reserved() / 2**20}
        history.append(row)
        with STEP_LOG.open('a', encoding='utf-8') as handle: handle.write(json.dumps(row) + '\n')
        if step % SAVE_EVERY == 0:
            torch.save({'step': step, 'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'scaler': scaler.state_dict(), 'settings': MODEL_SETTINGS.to_dict()}, CHECKPOINT)
    if time.perf_counter() - started >= BUDGET_SECONDS: break
torch.save({'step': step, 'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'scaler': scaler.state_dict(), 'settings': MODEL_SETTINGS.to_dict()}, CHECKPOINT)
print({'steps': step, 'wall_seconds': time.perf_counter() - started, 'tokens': tokens_seen, 'last_tokens_per_sec': history[-1]['tokens_per_sec'] if history else None})


In [ ]:
# Full validation metrics and expert-load diagnostics
def evaluate(loader, limit=None):
    model.eval(); total_loss = total_tokens = 0; all_assignments = []; times = []; surprises = []
    with torch.inference_mode():
        for index, batch in enumerate(loader):
            if limit is not None and index >= limit: break
            x = batch['input_ids'].to(DEVICE, non_blocking=True); y = batch['target_ids'].to(DEVICE, non_blocking=True); tm = batch['thinking_mask'].to(DEVICE, non_blocking=True)
            count = int((y != IGNORE_TARGET_ID).sum());
            if not count: continue
            t0 = time.perf_counter()
            with torch.autocast(device_type='cuda', dtype=AMP_DTYPE): out = model(x, execution_mode=ExecutionMode.PARALLEL); obj = calculate_training_objective(out, y, tm, 1.0)
            times.append(time.perf_counter() - t0); total_loss += float(obj.task_loss) * count; total_tokens += count
            all_assignments.append(out.active_expert_indices[out.active_expert_indices >= 0].flatten().cpu()); surprises.append(out.surprise_values[out.valid_positions].float().cpu())
    assignments = torch.cat(all_assignments) if all_assignments else torch.empty(0, dtype=torch.long)
    counts = torch.bincount(assignments, minlength=128).numpy() if assignments.numel() else np.zeros(128, dtype=np.int64)
    probabilities = counts / max(counts.sum(), 1); entropy = float(-(probabilities[probabilities > 0] * np.log(probabilities[probabilities > 0])).sum())
    mean_loss = total_loss / max(total_tokens, 1)
    result = {'loss_nats': mean_loss, 'bpb': mean_loss / math.log(2), 'answer_bpb': mean_loss / math.log(2), 'perplexity': math.exp(min(mean_loss, 80)), 'tokens': total_tokens, 'p50_batch_seconds': float(np.median(times)) if times else None, 'p95_batch_seconds': float(np.percentile(times, 95)) if times else None, 'tokens_per_sec': total_tokens / max(sum(times), 1e-9), 'expert_entropy_nats': entropy, 'expert_load_min': int(counts.min()), 'expert_load_max': int(counts.max()), 'expert_load_gini': float(np.abs(np.subtract.outer(counts, counts)).sum() / (2 * len(counts) * max(counts.sum(), 1)))}
    if surprises: result.update({'surprise_mean': float(torch.cat(surprises).mean()), 'surprise_p95': float(torch.quantile(torch.cat(surprises), .95))})
    model.train(); return result, counts
validation, expert_counts = evaluate(valid_loader)
(RESULTS / 'validation.json').write_text(json.dumps(validation, indent=2), encoding='utf-8')
print(json.dumps(validation, indent=2)); print({'expert_counts': expert_counts.tolist(), 'total_active_assignments': int(expert_counts.sum())})


In [ ]:
# Evidence plots: restrained white / pastel-yellow monochrome
plt.rcParams.update({'font.family': 'DejaVu Sans', 'text.color': '#202020', 'axes.labelcolor': '#202020', 'xtick.color': '#202020', 'ytick.color': '#202020'})
if history:
    frame = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), facecolor='white')
    axes[0].plot(frame.step, frame.bpb, color='#b08b00', linewidth=1.4); axes[0].set_title('Training byte-per-bit loss'); axes[0].set_xlabel('optimizer step'); axes[0].set_ylabel('bits/token')
    axes[1].plot(frame.step, frame.tokens_per_sec, color='#806800', linewidth=1.2); axes[1].set_title('Throughput'); axes[1].set_xlabel('optimizer step'); axes[1].set_ylabel('supervised tokens/s')
    axes[2].bar(np.arange(128), expert_counts, color='#e8d98a', edgecolor='#806800', linewidth=.25); axes[2].set_title('128-expert active assignment load'); axes[2].set_xlabel('expert'); axes[2].set_ylabel('assignments')
    fig.tight_layout(); fig.savefig(RESULTS / 'training_evidence.png', dpi=180, facecolor='white'); plt.show()
else: print('No training history in this session; validation and checkpoint files remain authoritative.')
print({'results_dir': str(RESULTS), 'checkpoint': str(CHECKPOINT), 'validation': str(RESULTS / 'validation.json')})


## How to read the result

For a supervised byte token, cross-entropy is `−log p(y|x)` in natural units. BPB is `loss / ln(2)`; perplexity is `exp(loss)`. Throughput is the number of supervised target bytes divided by measured wall time, not an estimate. With six active experts, the assignment total should be `6 × valid_tokens`; entropy, min/max load, and Gini expose whether deterministic hashing is uneven.

This experiment measures the current HERM implementation, not Transformer parity. The known trade-offs remain exact local detail versus compressed long-range memory, and deterministic dispatch versus a learned router. A good score is evidence for this configuration only; it is not a claim of superiority.